# Improving the model

🎯 **Goals**

- Improve on the baseline Random Forest model (notebook 4)
- Understand the impact of key hyperparameters on model performance
- Compare Random Forest with alternative models
- Evaluate whether increased complexity leads to meaningful performance gains 

### 🔁 Baseline Model Recap

The initial Random Forest model achieved:

- **MAE**: ~0.115  
- **RMSE**: ~0.209  
- **R²**: ~0.72  

This serves as the benchmark for further improvements.

### Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

### Loading and Splitting the data

In [2]:
data_dir = Path("../data")

# Load features
X = pd.read_csv(data_dir / "X.csv")

# Load target and convert df to series
y = pd.read_csv(data_dir / "y.csv").squeeze("columns")

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (17540, 405)
X_test: (4385, 405)
y_train: (17540,)
y_test: (4385,)


### Set up reporting

In [4]:
results = []

def evaluate_model(name, y_true, y_pred, params=None):
    """
    Evaluate a model and store results in a consistent format
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    results.append({
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "params": str(params) if params else ""
    })

    print(f"{name}")
    print(f"MAE: {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²: {r2:.3f}")
    print("-" * 30)

In [5]:
# Add RandomForest baseline results
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=1
)

rf_params = {
    "n_estimators": 200,
    "random_state": 42,
    "n_jobs": 1
}

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

evaluate_model("Random Forest (baseline)", y_test, rf_preds, rf_params)

results

Random Forest (baseline)
MAE: 0.115
RMSE: 0.209
R²: 0.725
------------------------------


[{'model': 'Random Forest (baseline)',
  'MAE': 0.11544490957131097,
  'RMSE': np.float64(0.20914985740426417),
  'R2': 0.7248203172551156,
  'params': "{'n_estimators': 200, 'random_state': 42, 'n_jobs': 1}"}]

## Random Forest Tuning

Random Forest performance is influenced by several key hyperparameters:

- `n_estimators`: number of trees (more = more stable, slower)
- `max_depth`: controls tree complexity (prevents overfitting)
- `min_samples_split`: minimum samples required to split a node (regularisation)

We explore whether tuning these improves performance beyond the baseline.

In [6]:
# Tuned version of Random Forest
rf_tuned = RandomForestRegressor(
    n_estimators=200,       # more trees → more stable predictions
    max_depth=10,           # limit depth → reduce overfitting
    min_samples_split=5,    # require more data to split → smoother model
    random_state=42,
    n_jobs=-1               # use all cores (faster)
)

rf_params_tuned = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_split": 5,
    "random_state": 42,
    "n_jobs": -1
}

# Train
rf_tuned.fit(X_train, y_train)

# Predict
rf_preds_tuned = rf_tuned.predict(X_test)

evaluate_model("Tuned Random Forest v1", y_test, rf_preds_tuned, rf_params_tuned)

Tuned Random Forest v1
MAE: 0.124
RMSE: 0.220
R²: 0.697
------------------------------


🙋‍♀️ **Tuned v1** was worse across all metrics. Regularisation (depth and split caps) don't allow for the complexity of the model, leading to underfitting. 

Next step: relax the constraints.

In [7]:
# More relaxed version of Random Forest
rf_v2 = RandomForestRegressor(
    n_estimators=300,   # increase stability
    max_depth=None,     # allow full depth again
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

rf_params_v2 = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "random_state": 42,
    "n_jobs": -1
}

# Train
rf_v2.fit(X_train, y_train)

# Predict
rf_preds_v2 = rf_v2.predict(X_test)

evaluate_model("Tuned Random Forest v2", y_test, rf_preds_v2, rf_params_v2)

Tuned Random Forest v2
MAE: 0.115
RMSE: 0.208
R²: 0.727
------------------------------


In [8]:
# Control leaf size
rf_v3 = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,   # prevents tiny noisy splits
    random_state=42,
    n_jobs=-1
)

rf_params_v3 = {
    "n_estimators": 300,
    "min_samples_leaf": 2,
    "random_state": 42,
    "n_jobs": -1
}

# Train
rf_v3.fit(X_train, y_train)

# Predict
rf_preds_v3 = rf_v3.predict(X_test)

evaluate_model("Tuned Random Forest v3", y_test, rf_preds_v3, rf_params_v3)

Tuned Random Forest v3
MAE: 0.115
RMSE: 0.209
R²: 0.726
------------------------------


### 💡 Random Forest tuning takeaways

- Constraining tree depth reduced performance, suggesting the problem benefits from flexible tree structures.

- Increasing the number of trees or lightly regularising leaf size produced almost identical results to the baseline.

- This indicates that the baseline Random Forest was already well-calibrated, and that further gains are unlikely to come from minor hyperparameter adjustments alone.

👉 At this point, **comparing different model families** is more informative than continuing to tune Random Forest.

## Gradient Boosting

Gradient Boosting is explored as an alternative to Random Forest, as it uses a different learning approach.

While Random Forest builds trees independently and averages their predictions, Gradient Boosting builds trees sequentially, with each new tree focusing on correcting errors made by the previous ones.

This makes it particularly effective at capturing subtle patterns and complex interactions in the data.

Given the weak individual feature relationships observed in EDA, but strong overall model performance, Gradient Boosting provides a useful comparison to assess whether more targeted learning improves predictive accuracy.

In [9]:
gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gbr_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 5,
    "random_state": 42
}

gbr.fit(X_train, y_train)
gbr_preds = gbr.predict(X_test)

evaluate_model("Gradient Boosting", y_test, gbr_preds, gbr_params)

Gradient Boosting
MAE: 0.114
RMSE: 0.199
R²: 0.750
------------------------------


### 💡 Gradient Boosting Results

Gradient Boosting achieves a modest improvement over the Random Forest baseline:
- **R²** increases from ~0.72 → ~0.75
- **RMSE** decreases, indicating improved handling of larger prediction errors
- **MAE** remains broadly similar

This suggests that sequential learning, where each tree focuses on correcting previous errors, is better suited to capturing the structure in this dataset than simple averaging.

The improvement is incremental rather than dramatic, indicating that much of the available signal was already captured by the Random Forest model.

👉 Overall, Gradient Boosting provides a small but meaningful performance gain.

## XGBoost

XGBoost is tested as a more advanced version of Gradient Boosting. It uses the same sequential error-correcting approach but introduces stronger regularisation and more efficient optimisation. It is particularly well-suited to structured/tabular data and is often able to extract additional performance beyond standard boosting methods.

In [10]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1
}

xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)

evaluate_model("XGBoost", y_test, xgb_preds, xgb_params)

XGBoost
MAE: 0.110
RMSE: 0.192
R²: 0.768
------------------------------


### 💡 XGBoost Results

XGBoost achieves the best performance across all tested models:
- **MAE**: ~0.110
- **RMSE**: ~0.192
- **R²**: ~0.768

This represents a consistent improvement over both Random Forest and Gradient Boosting.

The improvement suggests that a model which learns from its mistakes step by step performs better than one that simply averages many independent trees.

XGBoost also includes controls that help prevent overfitting, allowing it to capture useful patterns without becoming too sensitive to noise.

👉 **Overall, XGBoost provides the strongest performance and is selected as the final model**.

In [11]:
# XGB_v1 --> slower, more precise learning

xgb_v1 = XGBRegressor(
    n_estimators=500,      # increased
    learning_rate=0.03,    # decreased
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_v1_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1
}

xgb_v1.fit(X_train, y_train)
xgb_v1_preds = xgb_v1.predict(X_test)

evaluate_model("XGBoost v1", y_test, xgb_v1_preds, xgb_v1_params)

XGBoost v1
MAE: 0.110
RMSE: 0.191
R²: 0.770
------------------------------


In [ ]:
# XGB_v2 --> less complexity
# keep parameters that worked well from v1

xgb_v2 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,            # reduced
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_v2_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "max_depth": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1
}

xgb_v2.fit(X_train, y_train)
xgb_v2_preds = xgb_v2.predict(X_test)

evaluate_model("XGBoost v2", y_test, xgb_v2_preds, xgb_v2_params)

XGBoost v2
MAE: 0.114
RMSE: 0.198
R²: 0.753
------------------------------


In [ ]:
# XGB_v3 --> less complexity
# v2 worse so keep parameters that worked well from v1

xgb_v3 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,            # from v1
    subsample=0.7,          # reduced
    colsample_bytree=0.7,    # reduced
    random_state=42,
    n_jobs=-1
)

xgb_v3_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "max_depth": 6,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "random_state": 42,
    "n_jobs": -1
}

xgb_v3.fit(X_train, y_train)
xgb_v3_preds = xgb_v3.predict(X_test)

evaluate_model("XGBoost v3", y_test, xgb_v3_preds, xgb_v3_params)

XGBoost v3
MAE: 0.110
RMSE: 0.191
R²: 0.769
------------------------------


In [ ]:
# XGB_v4 --> more complexity

xgb_v4 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,            # increased
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_v4_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1
}

xgb_v4.fit(X_train, y_train)
xgb_v4_preds = xgb_v4.predict(X_test)

evaluate_model("XGBoost v4", y_test, xgb_v4_preds, xgb_v4_params)

XGBoost v4
MAE: 0.108
RMSE: 0.190
R²: 0.774
------------------------------


## results

In [16]:
results_df = pd.DataFrame(results)

results_df.sort_values(by="R2", ascending=False)

results_df.round(3)

,model,MAE,RMSE,R2,params
0,Random Forest (baseline),0.115,0.209,0.725,"{'n_estimators': 200, 'random_state': 42, 'n_j..."
1,Tuned Random Forest v1,0.124,0.220,0.697,"{'n_estimators': 200, 'max_depth': 10, 'min_sa..."
2,Tuned Random Forest v2,0.115,0.208,0.727,"{'n_estimators': 300, 'max_depth': None, 'min_..."
3,Tuned Random Forest v3,0.115,0.209,0.726,"{'n_estimators': 300, 'min_samples_leaf': 2, '..."
4,Gradient Boosting,0.114,0.199,0.750,"{'n_estimators': 200, 'learning_rate': 0.05, '..."
5,XGBoost,0.110,0.192,0.768,"{'n_estimators': 300, 'learning_rate': 0.05, '..."
6,XGBoost v1,0.110,0.191,0.770,"{'n_estimators': 500, 'learning_rate': 0.03, '..."
7,XGBoost v2,0.114,0.198,0.753,"{'n_estimators': 500, 'learning_rate': 0.03, '..."
8,XGBoost v3,0.110,0.191,0.769,"{'n_estimators': 500, 'learning_rate': 0.03, '..."
9,XGBoost v4,0.108,0.190,0.774,"{'n_estimators': 500, 'learning_rate': 0.03, '..."


## ⭐️ Final Model Selection

🏆 XGBoost achieved the best performance across all models tested:

- **MAE**: ~0.108
- **RMSE**: ~0.190
- **R²**: ~0.774

Performance improved consistently from Random Forest to Gradient Boosting to XGBoost, indicating that models which learn iteratively from previous errors are well suited to this problem.

The best XGBoost configuration used:

- more trees (n_estimators=500)
- a lower learning rate (learning_rate=0.03)
- deeper trees (max_depth=8)

This suggests the model benefits from:
- slower, more gradual learning
- enough tree depth to capture more complex feature interactions

A shallower version (max_depth=3) performed worse, indicating that oversimplifying the model reduces its ability to learn the structure in the data.

👉 Overall, the final model performs well because it combines careful step-by-step learning with enough flexibility to capture non-linear relationships.

🙋‍♀️ **Smaller steps helped, and trees that were too shallow missed important patterns.**